# Classification Model: Review Sentiment Prediction

**Team 9 — Recommender Systems Spring 2026**

## Problem Definition
This notebook implements a **binary classification model** that predicts whether a Yelp review is **positive or negative** based on the review text.

- **Positive (label = 1):** Reviews with 4 or 5 stars
- **Negative (label = 0):** Reviews with 1 or 2 stars
- **Excluded:** 3-star reviews are excluded as they are ambiguous and do not represent a clear sentiment signal

Review text is embedded using a pre-trained transformer model (`all-MiniLM-L6-v2` from `sentence-transformers`). A Logistic Regression classifier is then trained on these embeddings.

## Train/Test Data
- **Training data:** `train_reviews_santa_barbara.csv` (filtered to exclude 3-star reviews)
- **Test data:** `test_reviews_santa_barbara.csv` (filtered to exclude 3-star reviews)
- The provided train/test split is used as-is and has not been modified.

## Evaluation Metrics
Accuracy, Precision, Recall, F1, AUC

In [1]:
# Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

/Users/yureklio/Desktop/ReccomenderSystemsProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [2]:
def load_data(train_path: str, test_path: str):
    """
    Load and prepare train and test review datasets.

    Parameters
    ----------
    train_path : str
        Path to the training reviews CSV file.
    test_path : str
        Path to the test reviews CSV file.

    Returns
    -------
    tuple
        Two DataFrames: (train_df, test_df) with required columns.
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    for df in [train_df, test_df]:
        assert 'text' in df.columns, "Missing 'text' column"
        assert 'stars' in df.columns, "Missing 'stars' column"

    return train_df, test_df


train_raw, test_raw = load_data(
    'train_reviews_santa_barbara.csv',
    'test_reviews_santa_barbara.csv'
)

print(f'Raw train size: {len(train_raw)}')
print(f'Raw test size:  {len(test_raw)}')
print(f'\nTrain stars distribution:\n{train_raw["stars"].value_counts().sort_index()}')

Raw train size: 41581
Raw test size:  4801

Train stars distribution:
stars
1.0     2412
2.0     3149
3.0     5498
4.0    12446
5.0    18076
Name: count, dtype: int64


## 2. Build Train and Test Sets

We filter out 3-star (neutral) reviews to create a clear binary classification task.
- **Positive (1):** 4 or 5 stars
- **Negative (0):** 1 or 2 stars

In [3]:
def build_classification_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter reviews to positive/negative only and assign binary labels.

    Parameters
    ----------
    df : pd.DataFrame
        Raw reviews DataFrame with 'stars' and 'text' columns.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame with a new 'label' column (1=positive, 0=negative).
    """
    filtered = df[df['stars'] != 3].copy()
    filtered['label'] = (filtered['stars'] >= 4).astype(int)
    filtered['text'] = filtered['text'].fillna('')
    return filtered.reset_index(drop=True)


train_df = build_classification_dataset(train_raw)
test_df = build_classification_dataset(test_raw)

print(f'Train size after filtering: {len(train_df)}')
print(f'Test size after filtering:  {len(test_df)}')
print(f'\nTrain label distribution:\n{train_df["label"].value_counts()}')
print(f'\nTest label distribution:\n{test_df["label"].value_counts()}')

Train size after filtering: 36083
Test size after filtering:  4267

Train label distribution:
label
1    30522
0     5561
Name: count, dtype: int64

Test label distribution:
label
1    3564
0     703
Name: count, dtype: int64


## 3. Embed Review Text

We use the pre-trained `all-MiniLM-L6-v2` model from `sentence-transformers` to embed each review into a 384-dimensional vector. No fine-tuning is applied. For each review, the embedding represents the semantic meaning of the full review text.

These embeddings are used directly as input features for the classifier.

In [4]:
def embed_texts(texts: list, model_name: str = 'all-MiniLM-L6-v2', batch_size: int = 64) -> np.ndarray:
    """
    Embed a list of text strings using a pre-trained SentenceTransformer model.

    Parameters
    ----------
    texts : list of str
        List of review texts to embed.
    model_name : str
        Name of the pre-trained SentenceTransformer model to use.
    batch_size : int
        Batch size for encoding.

    Returns
    -------
    np.ndarray
        2D array of shape (n_samples, embedding_dim).
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=batch_size)
    return embeddings


print('Embedding training reviews...')
X_train = embed_texts(train_df['text'].tolist())

print('Embedding test reviews...')
X_test = embed_texts(test_df['text'].tolist())

y_train = train_df['label'].values
y_test = test_df['label'].values

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

Embedding training reviews...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7925.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 564/564 [01:53<00:00,  4.97it/s]


Embedding test reviews...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7239.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 67/67 [00:12<00:00,  5.41it/s]


X_train shape: (36083, 384)
X_test shape:  (4267, 384)


## 4. Train Classifier

We use **Logistic Regression** as our classifier. It is a well-established and interpretable model for binary classification tasks. Given high-dimensional embeddings as input, logistic regression learns a linear decision boundary in the embedding space that separates positive from negative reviews.

In [5]:
def train_classifier(X_train: np.ndarray, y_train: np.ndarray, random_state: int = 42) -> LogisticRegression:
    """
    Train a Logistic Regression classifier on the provided embeddings.

    Parameters
    ----------
    X_train : np.ndarray
        Training feature matrix (review embeddings).
    y_train : np.ndarray
        Binary labels (1=positive, 0=negative).
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    LogisticRegression
        Trained classifier.
    """
    clf = LogisticRegression(max_iter=1000, random_state=random_state)
    clf.fit(X_train, y_train)
    return clf


clf = train_classifier(X_train, y_train)
print('Classifier trained successfully.')

Classifier trained successfully.


## 5. Final Evaluation (Required Deliverable)

**This is the final cell of the notebook.** No training or model fitting occurs after this point.

Evaluation is performed exclusively on the test set. Required metrics: Accuracy, Precision, Recall, F1, AUC.

In [6]:
# Generate predictions on test set
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

# Compute required metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_prob)

print('Classification Model: Review Sentiment Prediction — Test Metrics')
print('=' * 60)
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'AUC:       {auc:.4f}')
print()
print('Detailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative (1-2 stars)', 'Positive (4-5 stars)']))
print()
print('Train/Test Data Summary:')
print(f'  Training samples: {len(train_df)} (positive: {int(y_train.sum())}, negative: {int((y_train==0).sum())})')
print(f'  Test samples:     {len(test_df)} (positive: {int(y_test.sum())}, negative: {int((y_test==0).sum())})')
print(f'  3-star reviews excluded from both train and test.')

Classification Model: Review Sentiment Prediction — Test Metrics
Accuracy:  0.9363
Precision: 0.9490
Recall:    0.9762
F1 Score:  0.9624
AUC:       0.9759

Detailed Classification Report:
                      precision    recall  f1-score   support

Negative (1-2 stars)       0.86      0.73      0.79       703
Positive (4-5 stars)       0.95      0.98      0.96      3564

            accuracy                           0.94      4267
           macro avg       0.90      0.86      0.88      4267
        weighted avg       0.93      0.94      0.93      4267


Train/Test Data Summary:
  Training samples: 36083 (positive: 30522, negative: 5561)
  Test samples:     4267 (positive: 3564, negative: 703)
  3-star reviews excluded from both train and test.
